In [1]:
# 1 - CREATE - 4_REPAIR_TEST_CASE_OUTPUT.txt - START

# TIME START - BEGIN
import time
TIME_START = time.time()
# TIME START - FINISH

# IMPORT - BEGIN
from ollama import chat, Client
from pathlib import Path
from pwn import *
context.log_level = 'critical'
from typing import Tuple
import subprocess, sys, operator, re, time
# IMPORT - FINISH

# PATH - BEGIN
FILE_PARTIAL = 'TEXT/REPAIR_TEST_CASE_RAW.txt'
FILE_WORKING_INPUT = 'TEXT/REPAIR_TEST_CASE_RAW_BATCH.txt'
FILE_FINAL_OUTPUT = 'TEXT/4_REPAIR_TEST_CASE_OUTPUT.txt'
FILE_PROMPT = 'TEXT/5_HISTORY_REPAIR_TEST_CASE.txt'
FILE_INPUT_TEST_CASE = 'TEXT/1_REPAIR_TEST_CASE_INPUT.txt'
# PATH - FINISH

# FUNCTION - BEGIN
def FUNCTION_PARSE_BLOCK_GLOBAL(STRING_TEXT: str):
    REGEX_PATTERN = re.compile(r'(#\s*(\d+)\s*# Extension Must Be Active:.*?)(?=\n\s*\n#\s*\d+\s*# Extension Must Be Active:|\Z)', re.DOTALL)
    MAP_BLOCK = {}
    for EACH_FULL, EACH_NUM in REGEX_PATTERN.findall(STRING_TEXT):
        MAP_BLOCK[int(EACH_NUM)] = EACH_FULL.strip()
    return MAP_BLOCK

def FUNCTION_WRITE_BLOCK_TO_FILE(MAP_BLOCK, LIST_NUMBER, FILE_OUTPUT_PATH):
    STRING_TEXT = '\n\n'.join(MAP_BLOCK[EACH_NUMBER] for EACH_NUMBER in LIST_NUMBER)
    with open(FILE_OUTPUT_PATH, 'w', encoding='utf-8', newline='\n') as FILE_HANDLE:
        FILE_HANDLE.write(STRING_TEXT.strip() + '\n')

def FUNCTION_READ_BLOCK_FROM_FILE(FILE_PATH):
    with open(FILE_PATH, 'r', encoding='utf-8') as FILE_HANDLE:
        return FUNCTION_PARSE_BLOCK_GLOBAL(FILE_HANDLE.read())

def FUNCTION_WRITE_ALL_BLOCK_SORTED(MAP_BLOCK, FILE_OUTPUT_PATH):
    STRING_TEXT = '\n\n'.join(MAP_BLOCK[EACH_KEY] for EACH_KEY in sorted(MAP_BLOCK))
    with open(FILE_OUTPUT_PATH, 'w', encoding='utf-8', newline='\n') as FILE_HANDLE:
        FILE_HANDLE.write(STRING_TEXT.strip() + '\n')

def FUNCTION_PLAY_QEMU():
    global ERROR_COUNT
    global FIRST_TIME
    def FUNCTION_PARSE_REGISTER(FILE_PATH):
        MAP_REGISTER = {}
        with open(FILE_PATH, 'r', encoding='utf-8') as FILE_HANDLE:
            for EACH_LINE in FILE_HANDLE:
                LIST_PART = EACH_LINE.split()
                if len(LIST_PART) >= SKIPPING_LINE_COUNT:
                    MAP_REGISTER[LIST_PART[0]] = LIST_PART[1]
        return MAP_REGISTER
    def FUNCTION_CLEAN_REGISTER_OUTPUT(STRING_OUTPUT):
        LIST_LINE = STRING_OUTPUT.splitlines()
        STRING_TRIMMED = '\n'.join(LIST_LINE[SKIPPING_LINE_COUNT:-1])
        return REGEX_ANSI_ESCAPE.sub('', STRING_TRIMMED)
    def FUNCTION_HANDLE_QEMU_ERROR(LIST_CMDLINE):
        RESULT_ERROR = subprocess.run(LIST_CMDLINE, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        print(RESULT_ERROR.stderr.strip())
    def FUNCTION_GET_CHANGED_REGISTER_INFO(IO_HANDLE, LIST_CHANGED_REGISTER):
        STRING_REGISTER = ' '.join(LIST_CHANGED_REGISTER)
        IO_HANDLE.sendline(f'info register {STRING_REGISTER}'.encode())
        STRING_OUTPUT = IO_HANDLE.clean().decode()
        LIST_LINE = STRING_OUTPUT.splitlines()
        if LIST_LINE and '(gdb)' in LIST_LINE[-1]:
            LIST_LINE.pop()
        return '\n'.join(LIST_LINE).strip()
    def FUNCTION_TEST_TEST_CASE(IO_HANDLE):
        MAP_REGISTER_OLD = FUNCTION_PARSE_REGISTER(FILE_REGISTER_DEFAULT)
        MAP_REGISTER_NEW = FUNCTION_PARSE_REGISTER(FILE_REGISTER_CHANGED)
        if ONLY_SHOW_CHANGED_REGISTER:
            LIST_CHANGED_REGISTER = []
            SET_EXCLUDED_REGISTER = {'sp', 'fp', 'pc', 'mcycle', 'minstret', 'cycle', 'time', 'instret'}
            for EACH_REGISTER, EACH_NEW_VALUE in MAP_REGISTER_NEW.items():
                STRING_OLD_VALUE = MAP_REGISTER_OLD.get(EACH_REGISTER)
                if STRING_OLD_VALUE and EACH_REGISTER not in SET_EXCLUDED_REGISTER and EACH_NEW_VALUE != STRING_OLD_VALUE:
                    LIST_CHANGED_REGISTER.append(EACH_REGISTER)
        else:
            LIST_CHANGED_REGISTER = LIST_REGISTER_CHOSEN
        if not LIST_CHANGED_REGISTER:
            STRING_OUTPUT = 'No registers changed!'
        else:
            STRING_OUTPUT = FUNCTION_GET_CHANGED_REGISTER_INFO(IO_HANDLE, LIST_CHANGED_REGISTER)
        print(STRING_OUTPUT)
    def FUNCTION_EDIT_MAIN_S(STRING_CONTENT):
        with open(FILE_CODE_BOOTLOADER, 'w', encoding='utf-8') as FILE_HANDLE:
            FILE_HANDLE.write(STRING_CONTENT)
    def FUNCTION_TIDY_OUTPUT_FILE(FILE_NAME):
        with open(FILE_NAME, 'r', encoding='utf-8') as FILE_HANDLE:
            LIST_LINE = FILE_HANDLE.readlines()
        VALUE_START = 0
        while VALUE_START < len(LIST_LINE) and not (LIST_LINE[VALUE_START].startswith('============') or 'START ITER' in LIST_LINE[VALUE_START]):
            VALUE_START += 1
        LIST_KEPT = LIST_LINE[VALUE_START:]
        LIST_COLLAPSED = []
        VALUE_BLANK_STREAK = 0
        for EACH_LINE in LIST_KEPT:
            if EACH_LINE.strip() == '':
                VALUE_BLANK_STREAK += 1
                if VALUE_BLANK_STREAK <= 1:
                    LIST_COLLAPSED.append(EACH_LINE)
            else:
                VALUE_BLANK_STREAK = 0
                LIST_COLLAPSED.append(EACH_LINE)
        while LIST_COLLAPSED and LIST_COLLAPSED[-1].strip() == '':
            LIST_COLLAPSED.pop()
        if LIST_COLLAPSED and LIST_COLLAPSED[-1].endswith('\n'):
            LIST_COLLAPSED[-1] = LIST_COLLAPSED[-1].rstrip('\n')
        with open(FILE_NAME, 'w', encoding='utf-8') as FILE_HANDLE:
            FILE_HANDLE.writelines(LIST_COLLAPSED)
    def FUNCTION_RUN_QEMU(FUNCTION_HOOK, FILE_REGISTER, MODE_VALUE):
        LIST_CMDLINE = ['./run', str(MODE_VALUE)]
        QEMU_IO = process(LIST_CMDLINE)
        try:
            QEMU_IO.recvuntil(b'*** Now run')
            IO_GDB = process('riscv64-unknown-elf-gdb')
            IO_GDB.sendline(b'b *test_end')
            IO_GDB.sendline(b'b *trap_vector')
            IO_GDB.sendline(b'c')
            STRING_OUTPUT = IO_GDB.recvuntil(b'Breakpoint 1, ', timeout=TIMEOUT_SECOND).decode()
            if not STRING_OUTPUT:
                IO_GDB.recvuntil(b'Breakpoint 2, ', timeout=TIMEOUT_SECOND).decode()
            IO_GDB.sendline(b'info all-registers')
            STRING_OUTPUT = IO_GDB.clean().decode()
            STRING_REGISTER_CLEANED = FUNCTION_CLEAN_REGISTER_OUTPUT(STRING_OUTPUT)
            with open(FILE_REGISTER, 'w', encoding='utf-8') as FILE_HANDLE:
                FILE_HANDLE.write(STRING_REGISTER_CLEANED)
            if FUNCTION_HOOK:
                FUNCTION_HOOK(IO_GDB)
            IO_GDB.kill()
        except EOFError:
            FUNCTION_HANDLE_QEMU_ERROR(LIST_CMDLINE)
        finally:
            QEMU_IO.kill()
    FILE_REGISTER_DEFAULT = 'TEXT/REPAIR_TEST_CASE_REGISTER_DEFAULT.txt'
    FILE_REGISTER_CHANGED = 'TEXT/REPAIR_TEST_CASE_REGISTER_CHANGE.txt'
    FILE_CODE_BOOTLOADER = 'bootloader/code.S'
    FILE_INPUT = 'TEXT/1_REPAIR_TEST_CASE_INPUT.txt'
    FILE_OUTPUT = 'TEXT/2_REPAIR_TEST_CASE_RUN.txt'
    MODE_DEFAULT = 3
    SKIPPING_LINE_COUNT = 3
    TIMEOUT_SECOND = 1
    STRING_DEFAULT = '''li x0, 0
li x1, 0
li x2, 0
li x3, 0
li x4, 0
li x5, 0
li x6, 0
li x7, 0
li x8, 0
li x9, 0
li x10, 0
li x11, 0
li x12, 0
li x13, 0
li x14, 0
li x15, 0
li x16, 0
li x17, 0
li x18, 0
li x19, 0
li x20, 0
li x21, 0
li x22, 0
li x23, 0
li x24, 0
li x25, 0
li x26, 0
li x27, 0
li x28, 0
li x29, 0
li x30, 0
li x31, 0
fmv.d.x f0, x0
fmv.d.x f1, x0
fmv.d.x f2, x0
fmv.d.x f3, x0
fmv.d.x f4, x0
fmv.d.x f5, x0
fmv.d.x f6, x0
fmv.d.x f7, x0
fmv.d.x f8, x0
fmv.d.x f9, x0
fmv.d.x f10, x0
fmv.d.x f11, x0
fmv.d.x f12, x0
fmv.d.x f13, x0
fmv.d.x f14, x0
fmv.d.x f15, x0
fmv.d.x f16, x0
fmv.d.x f17, x0
fmv.d.x f18, x0
fmv.d.x f19, x0
fmv.d.x f20, x0
fmv.d.x f21, x0
fmv.d.x f22, x0
fmv.d.x f23, x0
fmv.d.x f24, x0
fmv.d.x f25, x0
fmv.d.x f26, x0
fmv.d.x f27, x0
fmv.d.x f28, x0
fmv.d.x f29, x0
fmv.d.x f30, x0
fmv.d.x f31, x0
vsetvli x0, x0, e32, m1, ta, ma
vmv.v.i v0, 0
vmv.v.i v1, 0
vmv.v.i v2, 0
vmv.v.i v3, 0
vmv.v.i v4, 0
vmv.v.i v5, 0
vmv.v.i v6, 0
vmv.v.i v7, 0
vmv.v.i v8, 0
vmv.v.i v9, 0
vmv.v.i v10, 0
vmv.v.i v11, 0
vmv.v.i v12, 0
vmv.v.i v13, 0
vmv.v.i v14, 0
vmv.v.i v15, 0
vmv.v.i v16, 0
vmv.v.i v17, 0
vmv.v.i v18, 0
vmv.v.i v19, 0
vmv.v.i v20, 0
vmv.v.i v21, 0
vmv.v.i v22, 0
vmv.v.i v23, 0
vmv.v.i v24, 0
vmv.v.i v25, 0
vmv.v.i v26, 0
vmv.v.i v27, 0
vmv.v.i v28, 0
vmv.v.i v29, 0
vmv.v.i v30, 0
vmv.v.i v31, 0

'''
    REGEX_ANSI_ESCAPE = re.compile(r'\x1B\[[0-?]*[ -/]*[@-~]')
    ONLY_SHOW_CHANGED_REGISTER = False
    
    # IMPORTANT - BEGIN
    LIST_REGISTER_CHOSEN = ['priv', 'pc', 'a0', 'a1', 'a2', 'a3', 't0', 't1', 't2', 't3', 'f0', 'f1', 'f2', 'f3', 'ft0', 'ft1', 'ft2', 'ft3', 'v0', 'v1', 'v2', 'v3', 'x1', 'x2', 'x3', 'x4', 'x5', 'x6', 'x7', 'x8', 'x9', 'x10', 'x11', 'x12', 'x13', 'x14', 'x15', 'x16', 'x17', 'x18', 'x19', 'x20', 'x21', 'x22', 'x23', 'x24', 'x25', 'x26', 'x27', 'x28', 'x29', 'x30', 'x31', 'ra', 'sp', 'gp', 'tp', 'mstatus', 'mcause']
    IS_OUTPUT = False
    # IMPORTANT - FINISH
    
    STRING_CODE_TEMPLATE = '''
.globl test_start

test_start:
{test_case}

test_end:
j test_end

.global trap_vector
.align 4

trap_vector:
j trap_vector
    '''
    if IS_OUTPUT:
        class CLASS_STDOUT_TO_FILE:
            def __init__(self, FILE_NAME):
                self.file = open(FILE_NAME, 'w', encoding='utf-8')
                self.stdout = sys.__stdout__
            def write(self, STRING_MESSAGE):
                self.file.write(REGEX_ANSI_ESCAPE.sub('', STRING_MESSAGE))
                self.stdout.write(STRING_MESSAGE)
            def flush(self):
                self.file.flush()
                self.stdout.flush()
            def close(self):
                self.file.close()
    else:
        class CLASS_STDOUT_TO_FILE:
            def __init__(self, FILE_NAME):
                self.file = open(FILE_NAME, 'w', encoding='utf-8')
            def write(self, STRING_MESSAGE):
                self.file.write(REGEX_ANSI_ESCAPE.sub('', STRING_MESSAGE))
            def flush(self):
                self.file.flush()
            def close(self):
                self.file.close()
    OBJECT_STDOUT_FILE = CLASS_STDOUT_TO_FILE(FILE_OUTPUT)
    OBJECT_STDOUT_OLD = sys.stdout
    sys.stdout = OBJECT_STDOUT_FILE
    with open(FILE_INPUT, 'r', encoding='utf-8') as FILE_HANDLE:
        STRING_TEST_CASE = FILE_HANDLE.read()
    LIST_BLOCK = [EACH_BLOCK.strip().split('\n') for EACH_BLOCK in STRING_TEST_CASE.strip().split('\n\n') if EACH_BLOCK.strip()]
    LIST_TEST_CASE = [[EACH_LINE.strip() for EACH_LINE in EACH_BLOCK if EACH_LINE.strip()] for EACH_BLOCK in LIST_BLOCK]
    LIST_DEFAULT_TEST_CASE = [STRING_CODE_TEMPLATE.format(test_case=STRING_DEFAULT)]
    LIST_CHANGED_TEST_CASE = [STRING_CODE_TEMPLATE.format(test_case='\n'.join(STRING_DEFAULT.splitlines() + EACH_TEST_CASE) if EACH_TEST_CASE else STRING_DEFAULT) for EACH_TEST_CASE in LIST_TEST_CASE]
    try:
        FUNCTION_EDIT_MAIN_S(LIST_DEFAULT_TEST_CASE[0])
        FUNCTION_RUN_QEMU(None, FILE_REGISTER_DEFAULT, MODE_DEFAULT)
        for EACH_INDEX, (EACH_NEW_CONTENT, EACH_TEST_CASE) in enumerate(zip(LIST_CHANGED_TEST_CASE, LIST_TEST_CASE)):
            FUNCTION_EDIT_MAIN_S(EACH_NEW_CONTENT.strip())
            print(f'============\nSTART ITER {EACH_INDEX + 1}\n')
            print('\n'.join(EACH_TEST_CASE) + '\n')
            MODE_VALUE = MODE_DEFAULT
            FUNCTION_RUN_QEMU(FUNCTION_TEST_TEST_CASE, FILE_REGISTER_CHANGED, MODE_VALUE)
            if EACH_INDEX < len(LIST_CHANGED_TEST_CASE) - 1:
                print(f'\nEND ITER {EACH_INDEX + 1}\n==========\n')
            else:
                print(f'\nEND ITER {EACH_INDEX + 1}\n==========')
    finally:
        sys.stdout = OBJECT_STDOUT_OLD
        OBJECT_STDOUT_FILE.close()
        FUNCTION_TIDY_OUTPUT_FILE(FILE_OUTPUT)

def FUNCTION_CHECK_QEMU():
    global ERROR_COUNT
    global FIRST_TIME
    FILE_INPUT = 'TEXT/2_REPAIR_TEST_CASE_RUN.txt'
    FILE_OUTPUT = 'TEXT/3_REPAIR_TEST_CASE_CHECK.txt'
    def FUNCTION_CANONICAL_REGISTER(STRING_NAME: str) -> str:
        STRING_KEY = STRING_NAME.strip()
        MATCH_V = re.fullmatch(r'(v\d+)\[(\d+)\]', STRING_KEY)
        if MATCH_V:
            return f'{MATCH_V.group(1)}.w[{MATCH_V.group(2)}]'
        MATCH_V2 = re.fullmatch(r'(v\d+)\.(b|s|w|l|q)\[(\d+)\]', STRING_KEY)
        if MATCH_V2:
            return f'{MATCH_V2.group(1)}.{MATCH_V2.group(2)}[{MATCH_V2.group(3)}]'
        return MAP_REGISTER_ALIAS.get(STRING_KEY, STRING_KEY)
    def FUNCTION_PARSE_VECTOR_LIST(STRING_BODY: str, TOTAL_LEN: int):
        LIST_OUT = []
        TOKENS = [x.strip() for x in STRING_BODY.split(',') if x.strip()]
        for TOK in TOKENS:
            MREP = re.fullmatch(r'(0x[0-9a-fA-F]+)\s*<repeats\s+(\d+)\s+times>', TOK)
            if MREP:
                VAL = int(MREP.group(1), 16)
                CNT = int(MREP.group(2))
                LIST_OUT.extend([VAL] * CNT)
                continue
            MHEX = re.fullmatch(r'0x[0-9a-fA-F]+', TOK)
            if MHEX:
                LIST_OUT.append(int(TOK, 16))
                continue
            continue
        if len(LIST_OUT) < TOTAL_LEN:
            LIST_OUT.extend([0] * (TOTAL_LEN - len(LIST_OUT)))
        return LIST_OUT[:TOTAL_LEN]
    def FUNCTION_PARSE_INT(STRING_VALUE: str) -> int:
        STRING_LOWER = STRING_VALUE.strip().lower()
        if STRING_LOWER.startswith(('+0x', '-0x', '0x')):
            return int(STRING_VALUE, 16)
        return int(STRING_VALUE, 10)
    def FUNCTION_PARSE_ITERATION(STRING_CONTENT):
        REGEX_ITERATION = re.compile(r'START ITER (\d+)(.*?)END ITER \1', re.DOTALL)
        LIST_ITERATION = REGEX_ITERATION.findall(STRING_CONTENT)
        return LIST_ITERATION
    def FUNCTION_HAS_ERROR(STRING_ITERATION: str) -> bool:
        return re.search(r'\bpriv\b', STRING_ITERATION) is None
    def FUNCTION_PARSE_EXPECTED_RESULT(STRING_ITERATION: str):
        STRING_VALUE_PATTERN = r'[+-]?(?:0x[0-9a-fA-F]+|\d+)'
        STRING_REGISTER_PATTERN = r'(?:[A-Za-z_]\w+|v\d+(?:\.(?:b|s|w|l|q))?\[\d+\])'
        STRING_RHS_PATTERN = rf'(?:{STRING_VALUE_PATTERN}|{STRING_REGISTER_PATTERN})'
        REGEX_RESULT_LINE = re.compile(r'#\s*Result:\s*(.*)')
        REGEX_CONDITION = re.compile(
            rf'\s*({STRING_REGISTER_PATTERN})\s*(==|=|!=|<=|>=|<|>)\s*({STRING_RHS_PATTERN})\s*'
        )
        LIST_OR_GROUPS = []
        for EACH_LINE in STRING_ITERATION.splitlines():
            MATCH_RESULT = REGEX_RESULT_LINE.search(EACH_LINE)
            if not MATCH_RESULT:
                continue
            STRING_TAIL = MATCH_RESULT.group(1)
            LIST_OR_PART = [x.strip() for x in STRING_TAIL.split('|') if x.strip()]
            for EACH_OR in LIST_OR_PART:
                LIST_AND = []
                LIST_AND_PART = [x.strip() for x in EACH_OR.split('&') if x.strip()]
                for EACH_PART in LIST_AND_PART:
                    MATCH_COND = REGEX_CONDITION.fullmatch(EACH_PART)
                    if not MATCH_COND:
                        continue
                    STRING_REG, STRING_OP, STRING_RHS_TOKEN = MATCH_COND.groups()
                    STRING_REG = FUNCTION_CANONICAL_REGISTER(STRING_REG)
                    if STRING_OP == '=':
                        STRING_OP = '=='
                    LIST_AND.append((STRING_REG, STRING_OP, STRING_RHS_TOKEN))
                if LIST_AND:
                    LIST_OR_GROUPS.append(LIST_AND)
        return LIST_OR_GROUPS
    def FUNCTION_PARSE_ACTUAL_REGISTER(STRING_ITERATION: str):
        STRING_VALUE_PATTERN = r'[+-]?(?:0x[0-9a-fA-F]+|\d+)'
        REGEX_GPR = re.compile(rf'(\w+)\s+0x[0-9a-fA-F]+\s+({STRING_VALUE_PATTERN})')
        REGEX_FP_RAW = re.compile(
            r'^(f(?:t|s|a)?\d+)\s+[\s\S]*?\(raw\s+0x([0-9a-fA-F]+)\)',
            re.MULTILINE
        )
        REGEX_F_DOUBLE = re.compile(
            r'^(f\d+)\s+\{float\s*=\s*0x[0-9a-fA-F]+,\s*double\s*=\s*0x([0-9a-fA-F]+)\}',
            re.MULTILINE
        )
        MAP_ACTUAL = {}
        for EACH_REG, EACH_VALUE in REGEX_GPR.findall(STRING_ITERATION):
            try:
                VALUE_PARSED = FUNCTION_PARSE_INT(EACH_VALUE)
            except ValueError:
                continue
            STRING_CANON = FUNCTION_CANONICAL_REGISTER(EACH_REG)
            MAP_ACTUAL[STRING_CANON] = VALUE_PARSED
            MAP_ACTUAL[EACH_REG] = VALUE_PARSED
        for EACH_REG, EACH_HEX in REGEX_FP_RAW.findall(STRING_ITERATION):
            VALUE_PARSED = int(EACH_HEX, 16)
            STRING_CANON = FUNCTION_CANONICAL_REGISTER(EACH_REG)
            MAP_ACTUAL[STRING_CANON] = VALUE_PARSED
            MAP_ACTUAL[EACH_REG] = VALUE_PARSED
        for EACH_REG, EACH_HEX_DOUBLE in REGEX_F_DOUBLE.findall(STRING_ITERATION):
            VALUE_PARSED = int(EACH_HEX_DOUBLE, 16)
            STRING_CANON = FUNCTION_CANONICAL_REGISTER(EACH_REG)
            MAP_ACTUAL[STRING_CANON] = VALUE_PARSED
            MAP_ACTUAL[EACH_REG] = VALUE_PARSED
        REGEX_VBLOCK = re.compile(r'^(v\d+)\s+(.*?)(?=^v\d+\s+|\Z)', re.MULTILINE | re.DOTALL)
        REGEX_VIEW = re.compile(r'\b([qlwsb])\s*=\s*\{(.*?)\}', re.DOTALL)
        MAP_LEN = {'b': 16, 's': 8, 'w': 4, 'l': 2, 'q': 1}
        MAP_MASK = {'b': (1 << 8) - 1, 's': (1 << 16) - 1, 'w': (1 << 32) - 1, 'l': (1 << 64) - 1, 'q': MASK64}
        for VREG, VBLOCK in REGEX_VBLOCK.findall(STRING_ITERATION):
            VREG = VREG.strip()
            for VIEW, BODY in REGEX_VIEW.findall(VBLOCK):
                VIEW = VIEW.strip()
                if VIEW not in MAP_LEN:
                    continue
                LIST_VAL = FUNCTION_PARSE_VECTOR_LIST(BODY, MAP_LEN[VIEW])
                for IDX, VAL in enumerate(LIST_VAL):
                    KEY = f'{VREG}.{VIEW}[{IDX}]'
                    MAP_ACTUAL[KEY] = VAL & MAP_MASK[VIEW]
        return MAP_ACTUAL
    def FUNCTION_VECTOR_VIEW_BITS(STRING_REG_KEY: str):
        M = re.fullmatch(r'v\d+\.(b|s|w|l|q)\[\d+\]', STRING_REG_KEY)
        if not M:
            return None
        return {'b': 8, 's': 16, 'w': 32, 'l': 64, 'q': 64}[M.group(1)]
    def FUNCTION_HAS_RESULT_ONLY_ERROR(STRING_ITERATION: str) -> bool:
        return REGEX_RESULT_ONLY_ERROR.search(STRING_ITERATION) is not None
    def FUNCTION_U64(VALUE_X: int) -> int:
        return VALUE_X & MASK64
    def FUNCTION_CHECK_ITERATION(LIST_ITERATION):
        STRING_OUTPUT = ''
        for EACH_ITER_NUMBER, EACH_ITER_CONTENT in LIST_ITERATION:
            VALUE_ITER_NUMBER = int(EACH_ITER_NUMBER)
            if FUNCTION_HAS_RESULT_ONLY_ERROR(EACH_ITER_CONTENT):
                if FUNCTION_HAS_ERROR(EACH_ITER_CONTENT):
                    continue
                STRING_OUTPUT += f'START ITER {VALUE_ITER_NUMBER} is false\n'
                continue
            if FUNCTION_HAS_ERROR(EACH_ITER_CONTENT):
                STRING_OUTPUT += f'START ITER {VALUE_ITER_NUMBER} is error\n'
                continue
            LIST_EXPECTED_GROUP = FUNCTION_PARSE_EXPECTED_RESULT(EACH_ITER_CONTENT)
            MAP_ACTUAL = FUNCTION_PARSE_ACTUAL_REGISTER(EACH_ITER_CONTENT)
            FLAG_MATCH = False
            for EACH_GROUP in LIST_EXPECTED_GROUP:
                GROUP_OK = True
                for EACH_REG, EACH_OP, EACH_RHS_TOKEN in EACH_GROUP:
                    STRING_REG_KEY = FUNCTION_CANONICAL_REGISTER(EACH_REG)
                    VALUE_ACTUAL = MAP_ACTUAL.get(STRING_REG_KEY)
                    if VALUE_ACTUAL is None:
                        GROUP_OK = False
                        break
                    try:
                        VALUE_EXPECTED = FUNCTION_PARSE_INT(EACH_RHS_TOKEN)
                    except ValueError:
                        STRING_RHS_KEY = FUNCTION_CANONICAL_REGISTER(EACH_RHS_TOKEN)
                        VALUE_RHS = MAP_ACTUAL.get(STRING_RHS_KEY)
                        if VALUE_RHS is None:
                            GROUP_OK = False
                            break
                        VALUE_EXPECTED = VALUE_RHS
                    if EACH_OP in ('==', '!='):
                        WIDTH_BITS = None
                        if FUNCTION_IS_FLOAT_REG(STRING_REG_KEY):
                            WIDTH_BITS = FUNCTION_HEX_WIDTH_BITS(EACH_RHS_TOKEN)
                        V_BITS = FUNCTION_VECTOR_VIEW_BITS(STRING_REG_KEY)
                        if V_BITS is not None:
                            WIDTH_BITS = V_BITS
                        if WIDTH_BITS == 8:
                            VALUE_A = VALUE_ACTUAL & ((1 << 8) - 1)
                            VALUE_E = VALUE_EXPECTED & ((1 << 8) - 1)
                        elif WIDTH_BITS == 16:
                            VALUE_A = VALUE_ACTUAL & ((1 << 16) - 1)
                            VALUE_E = VALUE_EXPECTED & ((1 << 16) - 1)
                        elif WIDTH_BITS == 32:
                            VALUE_A = VALUE_ACTUAL & MASK32
                            VALUE_E = VALUE_EXPECTED & MASK32
                        else:
                            VALUE_A = FUNCTION_U64(VALUE_ACTUAL)
                            VALUE_E = FUNCTION_U64(VALUE_EXPECTED)
                        if not MAP_OPERATOR[EACH_OP](VALUE_A, VALUE_E):
                            GROUP_OK = False
                            break
                if GROUP_OK:
                    FLAG_MATCH = True
                    break
            if not FLAG_MATCH:
                STRING_OUTPUT += f'START ITER {VALUE_ITER_NUMBER} is false\n'
        STRING_OUTPUT_FINAL = STRING_OUTPUT.strip()
        with open(FILE_OUTPUT, 'w', encoding='utf-8') as FILE_HANDLE:
            FILE_HANDLE.write(STRING_OUTPUT_FINAL)
    def FUNCTION_IS_FLOAT_REG(STRING_REG_CANON: str) -> bool:
        return re.fullmatch(r'f(?:[0-9]|[12][0-9]|3[01])', STRING_REG_CANON) is not None
    def FUNCTION_HEX_WIDTH_BITS(STRING_TOKEN: str):
        S = STRING_TOKEN.strip().lower()
        if S.startswith(('+0x', '-0x')):
            S = S[1:]
        if not S.startswith('0x'):
            return None
        HEXDIG = S[2:]
        if 1 <= len(HEXDIG) <= 8:
            return 32
        if 9 <= len(HEXDIG) <= 16:
            return 64
        return None
    REGEX_RESULT_ONLY_ERROR = re.compile(r'^\s*#\s*Result\s*:\s*error\s*$', re.IGNORECASE | re.MULTILINE)
    MAP_OPERATOR = {'=': operator.eq, '==': operator.eq, '!=': operator.ne, '<': operator.lt, '<=': operator.le, '>': operator.gt, '>=': operator.ge}
    MAP_REGISTER_ALIAS = {'x0': 'x0', 'zero': 'x0', 'x1': 'x1', 'ra': 'x1', 'x2': 'x2', 'sp': 'x2', 'x3': 'x3', 'gp': 'x3', 'x4': 'x4', 'tp': 'x4', 'x5': 'x5', 't0': 'x5', 'x6': 'x6', 't1': 'x6', 'x7': 'x7', 't2': 'x7', 'x8': 'x8', 's0': 'x8', 'fp': 'x8', 'x9': 'x9', 's1': 'x9', 'x10': 'x10', 'a0': 'x10', 'x11': 'x11', 'a1': 'x11', 'x12': 'x12', 'a2': 'x12', 'x13': 'x13', 'a3': 'x13', 'x14': 'x14', 'a4': 'x14', 'x15': 'x15', 'a5': 'x15', 'x16': 'x16', 'a6': 'x16', 'x17': 'x17', 'a7': 'x17', 'x18': 'x18', 's2': 'x18', 'x19': 'x19', 's3': 'x19', 'x20': 'x20', 's4': 'x20', 'x21': 'x21', 's5': 'x21', 'x22': 'x22', 's6': 'x22', 'x23': 'x23', 's7': 'x23', 'x24': 'x24', 's8': 'x24', 'x25': 'x25', 's9': 'x25', 'x26': 'x26', 's10': 'x26', 'x27': 'x27', 's11': 'x27', 'x28': 'x28', 't3': 'x28', 'x29': 'x29', 't4': 'x29', 'x30': 'x30', 't5': 'x30', 'x31': 'x31', 't6': 'x31'}
    MASK64 = (1 << 64) - 1
    MASK32 = (1 << 32) - 1
    MAP_REGISTER_ALIAS.update({**{f'f{i}': f'f{i}' for i in range(32)}, 'ft0': 'f0', 'ft1': 'f1', 'ft2': 'f2', 'ft3': 'f3', 'ft4': 'f4', 'ft5': 'f5', 'ft6': 'f6', 'ft7': 'f7', 'ft8': 'f28', 'ft9': 'f29', 'ft10': 'f30', 'ft11': 'f31', 'fs0': 'f8', 'fs1': 'f9', 'fs2': 'f18', 'fs3': 'f19', 'fs4': 'f20', 'fs5': 'f21', 'fs6': 'f22', 'fs7': 'f23', 'fs8': 'f24', 'fs9': 'f25', 'fs10': 'f26', 'fs11': 'f27', 'fa0': 'f10', 'fa1': 'f11', 'fa2': 'f12', 'fa3': 'f13', 'fa4': 'f14', 'fa5': 'f15', 'fa6': 'f16', 'fa7': 'f17'})
    with open(FILE_INPUT, 'r', encoding='utf-8') as FILE_HANDLE:
        STRING_CONTENT = FILE_HANDLE.read()
    LIST_ITERATION = FUNCTION_PARSE_ITERATION(STRING_CONTENT)
    FUNCTION_CHECK_ITERATION(LIST_ITERATION)

def FUNCTION_FIX_THEM():
    global ERROR_COUNT
    global FIRST_TIME
    def FUNCTION_SPLIT_HEADER_BODY_FOOTER_TAIL(STRING_TEXT: str) -> Tuple[str, str, str, str]:
        STRING_TEXT = STRING_TEXT.replace('\r\n', '\n').strip('\n')
        LIST_LINE = STRING_TEXT.split('\n')
        STRING_HEADER = ''
        STRING_FOOTER = ''
        LIST_BODY_LINE = []
        LIST_TAIL_LINE = []
        VALUE_EXT_INDEX = None
        for EACH_INDEX, EACH_LINE in enumerate(LIST_LINE):
            if (re.match(r'^\s*#\s*\d+\s*#\s*Extension\b', EACH_LINE, re.IGNORECASE) or re.match(r'^\s*#\s*Extension\b', EACH_LINE, re.IGNORECASE)):
                VALUE_EXT_INDEX = EACH_INDEX
                break
        VALUE_RES_INDEX = None
        VALUE_START_SEARCH = (VALUE_EXT_INDEX + 1) if VALUE_EXT_INDEX is not None else 0
        for EACH_INDEX in range(VALUE_START_SEARCH, len(LIST_LINE)):
            if re.match(r'^\s*#\s*Result\b', LIST_LINE[EACH_INDEX], re.IGNORECASE):
                VALUE_RES_INDEX = EACH_INDEX
                break
        if VALUE_EXT_INDEX is None and VALUE_RES_INDEX is None:
            return '', STRING_TEXT.strip(), '', ''
        if VALUE_EXT_INDEX is not None:
            STRING_HEADER = LIST_LINE[VALUE_EXT_INDEX].strip()
        VALUE_BODY_START = (VALUE_EXT_INDEX + 1) if VALUE_EXT_INDEX is not None else 0
        VALUE_BODY_END = VALUE_RES_INDEX if VALUE_RES_INDEX is not None else len(LIST_LINE)
        LIST_BODY_LINE = LIST_LINE[VALUE_BODY_START:VALUE_BODY_END]
        if VALUE_RES_INDEX is not None:
            STRING_FOOTER = LIST_LINE[VALUE_RES_INDEX].strip()
            LIST_TAIL_LINE = LIST_LINE[VALUE_RES_INDEX + 1:]
        STRING_BODY = '\n'.join(LIST_BODY_LINE).strip('\n')
        LIST_TAIL_LINE = [EACH_LINE for EACH_LINE in LIST_TAIL_LINE if not EACH_LINE.startswith('None')]
        STRING_TAIL = '\n'.join(LIST_TAIL_LINE).strip('\n')
        return STRING_HEADER, STRING_BODY, STRING_FOOTER, STRING_TAIL
    def FUNCTION_CLEAN_ASSEMBLY(STRING_TEXT: str) -> str:
        LIST_LINE = STRING_TEXT.splitlines()
        LIST_CLEANED = [EACH_LINE.lstrip() for EACH_LINE in LIST_LINE]
        return '\n'.join(LIST_CLEANED)
    def FUNCTION_REMOVE_CODE_FENCE(STRING_TEXT: str) -> str:
        return re.sub(r'^\s*```.*\n?', '', STRING_TEXT, flags=re.MULTILINE)
    def FUNCTION_PARSE_BLOCK(STRING_TEXT: str):
        REGEX_PATTERN = re.compile(r'(#\s*(\d+)\s*# Extension Must Be Active:.*?)(?=\n\s*\n#\s*\d+\s*# Extension Must Be Active:|\Z)', re.DOTALL)
        MAP_BLOCK = {}
        for EACH_FULL, EACH_NUM in REGEX_PATTERN.findall(STRING_TEXT):
            MAP_BLOCK[int(EACH_NUM)] = EACH_FULL.strip()
        return MAP_BLOCK
    def FUNCTION_EXTRACT_NUMBER(STRING_BLOCK: str) -> int:
        MATCH_VALUE = re.search(r'#\s*(\d+)\s*# Extension Must Be Active:', STRING_BLOCK)
        if not MATCH_VALUE:
            raise ValueError('Cannot find block number')
        return int(MATCH_VALUE.group(1))
    def FUNCTION_NORMALIZE_NEW_BLOCK(STRING_BLOCK: str) -> str:
        STRING_BLOCK = re.sub(r'```[a-zA-Z]*\n?', '', STRING_BLOCK)
        STRING_BLOCK = STRING_BLOCK.replace('```', '')
        return STRING_BLOCK.strip()
    FILE_CHECK = 'TEXT/3_REPAIR_TEST_CASE_CHECK.txt'
    FILE_RUN = 'TEXT/2_REPAIR_TEST_CASE_RUN.txt'
    FILE_ORIGINAL = 'TEXT/REPAIR_TEST_CASE_RAW.txt'
    FILE_OUTPUT = 'TEXT/4_REPAIR_TEST_CASE_OUTPUT.txt'
    FILE_FIXING = 'TEXT/REPAIR_TEST_CASE_ATTEMPT.txt'
    FILE_BASE = FILE_OUTPUT if FIRST_TIME > 0 else FILE_ORIGINAL
    with open(FILE_BASE, 'r', encoding='utf-8') as FILE_HANDLE:
        STRING_OLD_TEXT = FILE_HANDLE.read()
    MAP_OLD_BLOCK = FUNCTION_PARSE_BLOCK(STRING_OLD_TEXT)
    SET_ERROR_ITER = set()
    VALUE_TEMP_ERROR = 0
    with open(FILE_CHECK, 'r', encoding='utf-8') as FILE_HANDLE:
        for EACH_LINE in FILE_HANDLE:
            EACH_LINE = EACH_LINE.strip()
            MATCH_VALUE = re.match(r'START ITER (\d+) is error', EACH_LINE)
            if MATCH_VALUE:
                SET_ERROR_ITER.add(int(MATCH_VALUE.group(1)))
                VALUE_TEMP_ERROR += 1
    if VALUE_TEMP_ERROR == 0:
        ERROR_COUNT = 0
        if FIRST_TIME == 0:
            with open(FILE_ORIGINAL, 'r', encoding='utf-8') as FILE_HANDLE:
                STRING_CONTENT = FILE_HANDLE.read().strip()
            with open(FILE_OUTPUT, 'w', encoding='utf-8') as FILE_HANDLE:
                FILE_HANDLE.write(STRING_CONTENT)
        return
    with open(FILE_RUN, 'r', encoding='utf-8') as FILE_HANDLE:
        STRING_TEXT = FILE_HANDLE.read()
    REGEX_ITER_PATTERN = re.compile(r'START ITER (\d+)\n(.*?)\nEND ITER \1', re.DOTALL)
    LIST_ERROR = []
    for MATCH_VALUE in REGEX_ITER_PATTERN.finditer(STRING_TEXT):
        VALUE_ITER = int(MATCH_VALUE.group(1))
        STRING_BODY = MATCH_VALUE.group(2).strip()
        if VALUE_ITER in SET_ERROR_ITER:
            STRING_BODY = re.sub(r'^=+\n?', '', STRING_BODY)
            STRING_BODY = re.sub(r'\n?=+$', '', STRING_BODY)
            LIST_ERROR.append(STRING_BODY.strip())
    MODEL = 'gpt-oss:120b'
    TEMPERATURE = 1
    CONTEXT = 16384
    API_KEY = '1dd95d27ffbc4b93a048c6ab1fbaa2e2.asxCCp5nO-KXM8jtg-TKNtw4'
    CLIENT = Client(host='https://ollama.com', headers={'Authorization': API_KEY})
    LIST_TOTAL_OUTPUT = []
    global GLOBAL_COUNTER
    global STRING_PROMPT_HISTORY
    for EACH_ERROR in LIST_ERROR:
        STRING_HEADER, STRING_BODY, STRING_FOOTER, STRING_TAIL = FUNCTION_SPLIT_HEADER_BODY_FOOTER_TAIL(EACH_ERROR)
        MATCH_NUMBER = re.search(r'#\s*(\d+)', STRING_BODY)
        MATCH_EXTENSION = re.search(r'#\s*Extension Must Be Active:.*', STRING_BODY)
        MATCH_RESULT = re.search(r'#\s*Result:.*', STRING_BODY)
        STRING_NUMBER = f'# {MATCH_NUMBER.group(1)}' if MATCH_NUMBER else None
        STRING_EXTENSION = MATCH_EXTENSION.group(0) if MATCH_EXTENSION else None
        STRING_RESULT = MATCH_RESULT.group(0) if MATCH_RESULT else None
        LIST_LINE = STRING_BODY.strip().splitlines()
        if LIST_LINE and LIST_LINE[0].startswith('#') and 'Extension Must Be Active:' in LIST_LINE[0]:
            LIST_LINE.pop(0)
        if LIST_LINE and LIST_LINE[-1].startswith('# Result:'):
            LIST_LINE.pop()
        STRING_BODY_CLEAN = '\n'.join(LIST_LINE)
        STRING_ERROR_MODIFIED = STRING_BODY_CLEAN + '\n\n' + 'THE ERROR:\n##########\n\n' + STRING_TAIL + '\n\n##########'
        STRING_PROMPT = f'''For RISC-V, minimally fix the assembly test case below while PRESERVING the ORIGINAL INSTRUCTION and OPERANDS whenever possible:

{STRING_ERROR_MODIFIED}

Do not output any introduction. Follow the format above.

Notes:
+ Remember, output must be ASCII-only text.
+ DO NOT USE .globl, _start, labels, directives, or PC-relative expressions; assume all boilerplate is already set up!
+ Do not use ```assembly or ```asm just write the assembly code
+ If the error message is empty, it usually means the program is hanging (e.g., infinite loop), so modify the test case to ensure it terminates.
+ Write the full assembly code, DO NOT write something like this:
/* ... other code ... */
# ... other existing code ...'''
        if IS_LOCAL:
            STRING_RESPONSE = chat(model=MODEL, messages=[{'role': 'user', 'content': STRING_PROMPT}], options={'temperature': TEMPERATURE, 'num_ctx': CONTEXT}).message.content.strip()
        else:
            STRING_RESPONSE = CLIENT.chat(model=MODEL, messages=[{'role': 'user', 'content': STRING_PROMPT}], options={'temperature': TEMPERATURE, 'num_ctx': CONTEXT}).message.content.strip()
        for EACH_CHARACTER in ['\u200b', '\u200c', '\u200d', '\ufeff']:
            STRING_RESPONSE = STRING_RESPONSE.replace(EACH_CHARACTER, '')
        for EACH_CHARACTER in ['\u00a0', '\u202f']:
            STRING_RESPONSE = STRING_RESPONSE.replace(EACH_CHARACTER, ' ')
        for EACH_RANGE in range(0xe000, 0xf900):
            STRING_RESPONSE = STRING_RESPONSE.replace(chr(EACH_RANGE), '')
        STRING_RESPONSE = STRING_RESPONSE.replace('“', '\'')
        STRING_RESPONSE = STRING_RESPONSE.replace('”', '\'')
        STRING_RESPONSE = STRING_RESPONSE.replace('←', '<-')
        STRING_RESPONSE = STRING_RESPONSE.replace('→', '->')
        STRING_RESPONSE = STRING_RESPONSE.replace('■', '+')
        PROMPT_INTRO_FIRST = f' START - {GLOBAL_COUNTER} '.center(WIDTH_PROMPT, '=')
        PROMPT_INTRO_END = f' END - {GLOBAL_COUNTER} '.center(WIDTH_PROMPT, '=')
        PROMPT_INTRO = f'''=======
PROMPT:
=======
{'=' * int(WIDTH_PROMPT / 2)}
'''
        PROMPT_ENDING = f'''
{'=' * int(WIDTH_PROMPT / 2)}
=========
RESPONSE:
=========
{'=' * int(WIDTH_PROMPT / 2)}
{STRING_RESPONSE}
'''
        STRING_PROMPT_HISTORY += (PROMPT_INTRO_FIRST + '\n' + PROMPT_INTRO + STRING_PROMPT + PROMPT_ENDING + PROMPT_INTRO_END + SUFFIX_FOR_PROMPT)
        GLOBAL_COUNTER += 1
        if not STRING_HEADER:
            MATCH_HEADER = re.search(r'^\s*#\s*\d+\s*#\s*Extension Must Be Active:.*$', EACH_ERROR, re.MULTILINE)
            if MATCH_HEADER:
                STRING_HEADER = MATCH_HEADER.group(0).strip()
        if not STRING_FOOTER:
            MATCH_FOOTER = re.search(r'^\s*#\s*Result\s*:.*$', EACH_ERROR, re.MULTILINE)
            if MATCH_FOOTER:
                STRING_FOOTER = MATCH_FOOTER.group(0).strip()
        STRING_NEW_RESULT = STRING_HEADER + '\n' + STRING_RESPONSE + '\n' + STRING_FOOTER
        STRING_FINAL = FUNCTION_REMOVE_CODE_FENCE(FUNCTION_CLEAN_ASSEMBLY(STRING_NEW_RESULT)).strip()
        STRING_FINAL = '\n'.join(EACH_LINE for EACH_LINE in STRING_FINAL.splitlines() if EACH_LINE.strip())
        LIST_TOTAL_OUTPUT.append(STRING_FINAL)
    STRING_FIXING_TEXT = '\n\n'.join(LIST_TOTAL_OUTPUT)
    with open(FILE_FIXING, 'w', encoding='utf-8') as FILE_HANDLE:
        FILE_HANDLE.write(STRING_FIXING_TEXT)
    for EACH_NEW_BLOCK in LIST_TOTAL_OUTPUT:
        VALUE_NUMBER = FUNCTION_EXTRACT_NUMBER(EACH_NEW_BLOCK)
        MAP_OLD_BLOCK[VALUE_NUMBER] = FUNCTION_NORMALIZE_NEW_BLOCK(EACH_NEW_BLOCK)
    STRING_FINAL_TEXT = '\n\n'.join(MAP_OLD_BLOCK[EACH_KEY] for EACH_KEY in sorted(MAP_OLD_BLOCK))
    with open(FILE_OUTPUT, 'w', encoding='utf-8') as FILE_HANDLE:
        FILE_HANDLE.write(STRING_FINAL_TEXT)
    with open('TEXT/1_REPAIR_TEST_CASE_INPUT.txt', 'w', encoding='utf-8', newline='\n') as FILE_HANDLE:
        FILE_HANDLE.write(STRING_FINAL_TEXT.strip() + '\n')
# FUNCTION - FINISH

# VARIABLE - BEGIN
GLOBAL_COUNTER = 1
IS_LOCAL = False
STRING_PROMPT_HISTORY = ''
WIDTH_PROMPT = 120
STRING_DIVIDE_FOR_PROMPT = ('#' * WIDTH_PROMPT + '\n') * 3
STRING_DIVIDE_FOR_PROMPT = STRING_DIVIDE_FOR_PROMPT.rstrip('\n')
SUFFIX_FOR_PROMPT = '\n' + STRING_DIVIDE_FOR_PROMPT + '\n'
MAP_ALL_BLOCK = FUNCTION_READ_BLOCK_FROM_FILE(FILE_PARTIAL)
MAP_FINAL_BLOCK = dict(MAP_ALL_BLOCK)
LIST_NUMBER = sorted(MAP_ALL_BLOCK.keys())
BATCH_SIZE = len(LIST_NUMBER)
FILE_A = FILE_INPUT_TEST_CASE
FILE_B = FILE_FINAL_OUTPUT
FILE_C = FILE_PROMPT
FILE_D = FILE_WORKING_INPUT
PATH_A = Path(FILE_A)
PATH_B = Path(FILE_B)
PATH_C = Path(FILE_C)
PATH_D = Path(FILE_D)
MAXIMUM_LIMIT = 10
# VARIABLE - FINISH

# CODE - BEGIN
with open(FILE_PROMPT, 'w', encoding='utf-8') as FILE_HANDLE:
    FILE_HANDLE.write('')

for EACH_BATCH_START in range(0, len(LIST_NUMBER), BATCH_SIZE):
    LIST_BATCH_NUMBER = LIST_NUMBER[EACH_BATCH_START:EACH_BATCH_START + BATCH_SIZE]
    FUNCTION_WRITE_BLOCK_TO_FILE(MAP_ALL_BLOCK, LIST_BATCH_NUMBER, FILE_WORKING_INPUT)
    FILE_INPUT = FILE_WORKING_INPUT
    FILE_OUTPUT = 'TEXT/1_REPAIR_TEST_CASE_INPUT.txt'
    with open(FILE_INPUT, 'r', encoding='utf-8') as FILE_HANDLE:
        STRING_CONTENT = FILE_HANDLE.read()
    with open(FILE_OUTPUT, 'w', encoding='utf-8') as FILE_HANDLE:
        FILE_HANDLE.write(STRING_CONTENT)
    ERROR_COUNT = -1
    FIRST_TIME = 0
    while ERROR_COUNT != 0 and FIRST_TIME < MAXIMUM_LIMIT:
        FUNCTION_PLAY_QEMU()
        FUNCTION_CHECK_QEMU()
        FUNCTION_FIX_THEM()
        FIRST_TIME += 1
    STRING_PROMPT_HISTORY = STRING_PROMPT_HISTORY.removesuffix(SUFFIX_FOR_PROMPT)
    with open(FILE_PROMPT, 'a', encoding='utf-8') as FILE_HANDLE:
        FILE_HANDLE.write(STRING_PROMPT_HISTORY + '\n\n')
    MAP_FIXED_BATCH_BLOCK = FUNCTION_READ_BLOCK_FROM_FILE('TEXT/4_REPAIR_TEST_CASE_OUTPUT.txt')
    for EACH_NUMBER in LIST_BATCH_NUMBER:
        if EACH_NUMBER in MAP_FIXED_BATCH_BLOCK:
            MAP_FINAL_BLOCK[EACH_NUMBER] = MAP_FIXED_BATCH_BLOCK[EACH_NUMBER]
        else:
            pass
    FUNCTION_WRITE_ALL_BLOCK_SORTED(MAP_FINAL_BLOCK, FILE_FINAL_OUTPUT)

LIST_LINE = PATH_A.read_text(encoding='utf-8').splitlines()

while LIST_LINE and not LIST_LINE[-1].strip():
    LIST_LINE.pop()

PATH_A.write_text('\n'.join(LIST_LINE), encoding='utf-8')
LIST_LINE = PATH_B.read_text(encoding='utf-8').splitlines()

while LIST_LINE and not LIST_LINE[-1].strip():
    LIST_LINE.pop()

PATH_B.write_text('\n'.join(LIST_LINE), encoding='utf-8')
LIST_LINE = PATH_C.read_text(encoding='utf-8').splitlines()

while LIST_LINE and not LIST_LINE[-1].strip():
    LIST_LINE.pop()

PATH_C.write_text('\n'.join(LIST_LINE), encoding='utf-8')
LIST_LINE = PATH_D.read_text(encoding='utf-8').splitlines()

while LIST_LINE and not LIST_LINE[-1].strip():
    LIST_LINE.pop()

PATH_D.write_text('\n'.join(LIST_LINE), encoding='utf-8')
# CODE - FINISH

# TIME END - BEGIN
TIME_END = time.time()
DURATION_IN_SECOND = TIME_END - TIME_START
DAYS = int(DURATION_IN_SECOND // 86400)
DURATION_IN_SECOND %= 86400
HOURS = int(DURATION_IN_SECOND // 3600)
DURATION_IN_SECOND %= 3600
MINUTES = int(DURATION_IN_SECOND // 60)
DURATION_IN_SECOND %= 60
PARTS = []
if DAYS:
    PARTS.append(f'{DAYS} day(s)')
if HOURS:
    PARTS.append(f'{HOURS} hour(s)')
if MINUTES:
    PARTS.append(f'{MINUTES} minute(s)')
if DURATION_IN_SECOND or not PARTS:
    PARTS.append(f'{DURATION_IN_SECOND:.2f} second(s)')
print(' '.join(PARTS))
# TIME END - FINISH

# DONE - BEGIN
print('====\nDONE\n====')
# DONE - FINISH

# 1 - CREATE - 4_REPAIR_TEST_CASE_OUTPUT.txt - END

16 minute(s) 14.21 second(s)
====
DONE
====


In [2]:
# 2 - UPDATE - 4_REPAIR_TEST_CASE_OUTPUT.txt - START

# TIME START - BEGIN
import time
TIME_START = time.time()
# TIME START - FINISH

# IMPORT - BEGIN
from pathlib import Path
import re
# IMPORT - FINISH

import shutil
file_a = 'TEXT/4_REPAIR_TEST_CASE_OUTPUT.txt'
file_b = 'TEXT/4_REPAIR_TEST_CASE_OUTPUT_1.txt'
shutil.copyfile(file_a, file_b)

# PATH - BEGIN
FILE_INPUT = Path('TEXT/4_REPAIR_TEST_CASE_OUTPUT_1.txt')
FILE_OUTPUT = Path('TEXT/4_REPAIR_TEST_CASE_OUTPUT_2.txt')
FILE_REGISTER = Path('TEXT/REPAIR_TEST_CASE_REGISTER.txt')
# PATH - FINISH

# FUNCTION - BEGIN
def FUNCTION_TWOS_SIGNED(VALUE_INT: int, BIT_COUNT: int) -> int:
    VALUE_MASK = (1 << BIT_COUNT) - 1
    VALUE_INT &= VALUE_MASK
    VALUE_SIGN = 1 << (BIT_COUNT - 1)
    return VALUE_INT - (1 << BIT_COUNT) if (VALUE_INT & VALUE_SIGN) else VALUE_INT

def FUNCTION_CONVERT_HEX_IN_LINE(STRING_LINE: str) -> str:
    def FUNCTION_REPLACE(MATCH_VALUE: re.Match) -> str:
        STRING_HEX = MATCH_VALUE.group(1)
        IS_NEG = STRING_HEX.startswith('-')
        STRING_HEX_BODY = STRING_HEX[1:] if IS_NEG else STRING_HEX
        VALUE_INT = int(STRING_HEX_BODY, 16)
        if IS_NEG:
            VALUE_INT = -VALUE_INT
        if REGEX_RESULT_HEADER.match(STRING_LINE):
            return str(FUNCTION_TWOS_SIGNED(VALUE_INT, RESULT_BIT))
        return str(VALUE_INT)
    return REGEX_HEX.sub(FUNCTION_REPLACE, STRING_LINE)

def FUNCTION_REPLACE_REGISTER(STRING_LINE: str) -> str:
    STRING_LINE = REGEX_X.sub(lambda MATCH_VALUE: MAP_X_ABI.get(MATCH_VALUE.group(0), MATCH_VALUE.group(0)), STRING_LINE)
    STRING_LINE = REGEX_F.sub(lambda MATCH_VALUE: MAP_F_ABI.get(MATCH_VALUE.group(0), MATCH_VALUE.group(0)), STRING_LINE)
    return STRING_LINE

def FUNCTION_REPLACE_CSR_NUMBER(STRING_LINE: str) -> str:
    STRING_STRIPPED = STRING_LINE.lstrip()
    if not STRING_STRIPPED or STRING_STRIPPED.startswith('#'):
        return STRING_LINE
    MATCH_OP = re.match(r'^(\s*)([A-Za-z.][A-Za-z0-9_.]*)\s+(.*)$', STRING_LINE)
    if not MATCH_OP:
        return STRING_LINE
    STRING_INDENT = MATCH_OP.group(1)
    STRING_OP = MATCH_OP.group(2).lower()
    STRING_REST = MATCH_OP.group(3)
    if STRING_OP not in SET_CSR_OP:
        return STRING_LINE
    NL = '\n' if STRING_LINE.endswith('\n') else ''
    STRING_BEFORE_HASH, STRING_COMMENT_TEXT = (STRING_REST.split('#', 1) + [''])[:2]
    STRING_BEFORE_HASH = STRING_BEFORE_HASH.strip()
    STRING_COMMENT_TEXT = STRING_COMMENT_TEXT.strip()
    LIST_OPERAND = [x.strip() for x in STRING_BEFORE_HASH.split(',') if x.strip()]
    INDEX_CSR = None
    if STRING_OP in SET_CSR_PRR or STRING_OP in SET_CSR_PI:
        INDEX_CSR = 0 if len(LIST_OPERAND) >= 1 else None
    elif STRING_OP in SET_CSR_RRR or STRING_OP in SET_CSR_RRI:
        INDEX_CSR = 1 if len(LIST_OPERAND) >= 2 else None
    elif STRING_OP == 'csrr':
        INDEX_CSR = 1 if len(LIST_OPERAND) >= 2 else None
    if INDEX_CSR is not None:
        STRING_TOKEN = LIST_OPERAND[INDEX_CSR]
        if STRING_TOKEN.lower().startswith('0x') or re.fullmatch(r'\d+', STRING_TOKEN):
            VALUE_NUM = int(STRING_TOKEN, 16) if STRING_TOKEN.lower().startswith('0x') else int(STRING_TOKEN, 10)
            STRING_NAME = MAP_CSR_NUM_TO_NAME.get(VALUE_NUM)
            if STRING_NAME:
                LIST_OPERAND[INDEX_CSR] = STRING_NAME
    STRING_NEW = STRING_INDENT + STRING_OP + ' ' + ', '.join(LIST_OPERAND)
    if '#' in STRING_REST:
        if STRING_COMMENT_TEXT:
            return STRING_NEW + ' # ' + STRING_COMMENT_TEXT + NL
        return STRING_NEW + ' #' + NL
    return STRING_NEW + NL

def FUNCTION_COLLAPSE_MCAUSE_BLOCK(STRING_TEXT: str) -> str:
    LIST_PART = re.split(r'(?m)(?=^#\s*\d+\s*#\s*Extension Must Be Active:)', STRING_TEXT)
    LIST_OUTPUT = []
    REGEX_CSRR_MCAUSE = re.compile(r'^\s*csrr\s+([A-Za-z0-9_]+)\s*,\s*mcause\s*(?:#.*)?\s*$', re.I)
    REGEX_CSRRS_MCAUSE = re.compile(r'^\s*csrrs\s+([A-Za-z0-9_]+)\s*,\s*mcause\s*,\s*(x0|zero)\s*(?:#.*)?\s*$', re.I)
    REGEX_RESULT_REG = re.compile(r'^\s*#\s*Result:\s*([A-Za-z0-9_]+)\s*=\s*(.+?)\s*$', re.I)
    for EACH_PART in LIST_PART:
        if not EACH_PART.strip():
            LIST_OUTPUT.append(EACH_PART)
            continue
        LIST_LINE = EACH_PART.splitlines(True)
        REGISTER_DEST = None
        for EACH_INDEX in range(len(LIST_LINE) - 1, -1, -1):
            MATCH_MCAUSE = REGEX_CSRR_MCAUSE.match(LIST_LINE[EACH_INDEX]) or REGEX_CSRRS_MCAUSE.match(LIST_LINE[EACH_INDEX])
            if MATCH_MCAUSE:
                REGISTER_DEST = MATCH_MCAUSE.group(1)
                break
        if REGISTER_DEST is None:
            LIST_OUTPUT.append(EACH_PART)
            continue
        VALUE_REMOVED = None
        LIST_NEW_LINE = []
        for EACH_LINE in LIST_LINE:
            MATCH_RESULT = REGEX_RESULT_REG.match(EACH_LINE)
            if MATCH_RESULT and MATCH_RESULT.group(1) == REGISTER_DEST:
                if VALUE_REMOVED is None:
                    VALUE_REMOVED = MATCH_RESULT.group(2).strip()
                continue
            LIST_NEW_LINE.append(EACH_LINE)
        if VALUE_REMOVED is None:
            LIST_OUTPUT.append(EACH_PART)
            continue
        LIST_FILTERED = []
        REMOVED_CSRR = False
        for EACH_LINE in LIST_NEW_LINE:
            if (not REMOVED_CSRR) and (REGEX_CSRR_MCAUSE.match(EACH_LINE) or REGEX_CSRRS_MCAUSE.match(EACH_LINE)):
                REMOVED_CSRR = True
                continue
            LIST_FILTERED.append(EACH_LINE)
        STRING_INSERT = f'# Result: mcause = {VALUE_REMOVED}\n'
        VALUE_J = len(LIST_FILTERED)
        while VALUE_J > 0 and LIST_FILTERED[VALUE_J - 1].strip() == '':
            VALUE_J -= 1
        LIST_FILTERED.insert(VALUE_J, STRING_INSERT)
        LIST_OUTPUT.append(''.join(LIST_FILTERED))
    return ''.join(LIST_OUTPUT)
# FUNCTION - FINISH

# VARIABLE - BEGIN
RESULT_BIT = 32
MAP_X_ABI = {'x0': 'zero','x1': 'ra','x2': 'sp','x3': 'gp','x4': 'tp', 'x5': 't0','x6': 't1','x7': 't2','x8': 's0','x9': 's1', 'x10': 'a0','x11': 'a1','x12': 'a2','x13': 'a3','x14': 'a4','x15': 'a5','x16': 'a6','x17': 'a7', 'x18': 's2','x19': 's3','x20': 's4','x21': 's5','x22': 's6','x23': 's7','x24': 's8','x25': 's9','x26': 's10','x27': 's11', 'x28': 't3','x29': 't4','x30': 't5','x31': 't6'}
MAP_F_ABI = {'f0':'ft0','f1':'ft1','f2':'ft2','f3':'ft3','f4':'ft4','f5':'ft5','f6':'ft6','f7':'ft7', 'f8':'fs0','f9':'fs1', 'f10':'fa0','f11':'fa1','f12':'fa2','f13':'fa3','f14':'fa4','f15':'fa5','f16':'fa6','f17':'fa7', 'f18':'fs2','f19':'fs3','f20':'fs4','f21':'fs5','f22':'fs6','f23':'fs7','f24':'fs8','f25':'fs9','f26':'fs10','f27':'fs11', 'f28':'ft8','f29':'ft9','f30':'ft10','f31':'ft11'}
REGEX_X = re.compile(r'\bx(3[01]|[12]\d|[0-9])\b')
REGEX_F = re.compile(r'\bf(3[01]|[12]\d|[0-9])\b')
REGEX_HEX = re.compile(r'(?<![A-Za-z0-9_])(-?0x[0-9a-fA-F]+)(?![A-Za-z0-9_])')
REGEX_RESULT_HEADER = re.compile(r'^\s*#\s*Result\s*:', re.IGNORECASE)
SET_CSR_RRR = {'csrrw', 'csrrs', 'csrrc'}
SET_CSR_RRI = {'csrrwi', 'csrrsi', 'csrrci'}
SET_CSR_PRR = {'csrw', 'csrs', 'csrc'}
SET_CSR_PI = {'csrwi', 'csrsi', 'csrci'}
SET_CSR_PSEUDO = {'csrr'}
SET_CSR_OP = SET_CSR_RRR | SET_CSR_RRI | SET_CSR_PRR | SET_CSR_PI | SET_CSR_PSEUDO
MAP_CSR_NUM_TO_NAME = {}
REGISTER_CURRENT = None
STRING_TEXT = FILE_INPUT.read_text(encoding='utf-8', errors='replace').replace('\r\n', '\n')
LIST_LINE = STRING_TEXT.splitlines(True)
LIST_OUTPUT_LINE = []
# VARIABLE - FINISH

# CODE - BEGIN
for EACH_LINE in FILE_REGISTER.read_text(encoding='utf-8', errors='replace').splitlines():
    MATCH_REGISTER = re.match(r'^\s*REGISTER:\s*([A-Za-z0-9_]+)\s*$', EACH_LINE)
    if MATCH_REGISTER:
        REGISTER_CURRENT = MATCH_REGISTER.group(1)
        continue
    MATCH_NUMBER = re.match(r'^\s*NUMBER:\s*(0x[0-9a-fA-F]+|\d+)\s*$', EACH_LINE)
    if MATCH_NUMBER and REGISTER_CURRENT:
        STRING_NUMBER = MATCH_NUMBER.group(1)
        VALUE_NUMBER = int(STRING_NUMBER, 16) if STRING_NUMBER.lower().startswith('0x') else int(STRING_NUMBER, 10)
        MAP_CSR_NUM_TO_NAME.setdefault(VALUE_NUMBER, REGISTER_CURRENT)
        REGISTER_CURRENT = None

for EACH_LINE in LIST_LINE:
    EACH_LINE = FUNCTION_REPLACE_CSR_NUMBER(EACH_LINE)
    EACH_LINE = FUNCTION_REPLACE_REGISTER(EACH_LINE)
    LIST_OUTPUT_LINE.append(EACH_LINE)

STRING_FINAL = ''.join(LIST_OUTPUT_LINE)
STRING_FINAL = FUNCTION_COLLAPSE_MCAUSE_BLOCK(STRING_FINAL)
FILE_OUTPUT.write_text(STRING_FINAL, encoding='utf-8', newline='\n')
# CODE - FINISH

# TIME END - BEGIN
TIME_END = time.time()
DURATION_IN_SECOND = TIME_END - TIME_START
DAYS = int(DURATION_IN_SECOND // 86400)
DURATION_IN_SECOND %= 86400
HOURS = int(DURATION_IN_SECOND // 3600)
DURATION_IN_SECOND %= 3600
MINUTES = int(DURATION_IN_SECOND // 60)
DURATION_IN_SECOND %= 60
PARTS = []
if DAYS:
    PARTS.append(f'{DAYS} day(s)')
if HOURS:
    PARTS.append(f'{HOURS} hour(s)')
if MINUTES:
    PARTS.append(f'{MINUTES} minute(s)')
if DURATION_IN_SECOND or not PARTS:
    PARTS.append(f'{DURATION_IN_SECOND:.2f} second(s)')
print(' '.join(PARTS))
# TIME END - FINISH

# DONE - BEGIN
print('====\nDONE\n====')
# DONE - FINISH

# 2 - UPDATE - 4_REPAIR_TEST_CASE_OUTPUT.txt - END

0.02 second(s)
====
DONE
====


In [3]:
# 3 - CREATE - REPAIR_TEST_CASE_OUTPUT_EXTENSION.txt - START

# TIME START - BEGIN
import time
TIME_START = time.time()
# TIME START - FINISH

# IMPORT - BEGIN
from pathlib import Path
import re
# IMPORT - FINISH

# PATH - BEGIN
FILE_INPUT = Path('TEXT/4_REPAIR_TEST_CASE_OUTPUT_2.txt')
FILE_OUTPUT = Path('TEXT/REPAIR_TEST_CASE_OUTPUT_EXTENSION.txt')
# PATH - FINISH

# FUNCTION - BEGIN
def FUNCTION_FILTER_EXTENSION_BLOCK(LIST_LINE):
    LIST_OUTPUT = []
    LIST_CURRENT = []
    CURRENT_EXTENSION = None
    for EACH_LINE in LIST_LINE:
        MATCH = REGEX_EXTENSION_HEADER.match(EACH_LINE)
        if MATCH:
            if (
                LIST_CURRENT
                and CURRENT_EXTENSION
                and CURRENT_EXTENSION.lower() != 'none'
            ):
                while LIST_CURRENT and LIST_CURRENT[-1].strip() == '':
                    LIST_CURRENT.pop()
                if LIST_OUTPUT:
                    LIST_OUTPUT.append('\n')
                LIST_OUTPUT.extend(LIST_CURRENT)
            LIST_CURRENT = [EACH_LINE]
            CURRENT_EXTENSION = MATCH.group('ext')
        else:
            LIST_CURRENT.append(EACH_LINE)
    if (
        LIST_CURRENT
        and CURRENT_EXTENSION
        and CURRENT_EXTENSION.lower() != 'none'
    ):
        while LIST_CURRENT and LIST_CURRENT[-1].strip() == '':
            LIST_CURRENT.pop()
        if LIST_OUTPUT:
            LIST_OUTPUT.append('\n')
        LIST_OUTPUT.extend(LIST_CURRENT)
    return LIST_OUTPUT
# FUNCTION - FINISH

# VARIABLE - BEGIN
REGEX_EXTENSION_HEADER = re.compile(
    r'^\s*#\s*\d+\s*#\s*Extension Must Be Active:\s*(?P<ext>\S+)',
    re.IGNORECASE
)
# VARIABLE - FINISH

# CODE - BEGIN
with open(FILE_INPUT, 'r', encoding='utf-8') as f:
    LIST_LINE = f.readlines()

LIST_OUTPUT = FUNCTION_FILTER_EXTENSION_BLOCK(LIST_LINE)
TEXT_OUTPUT = ''.join(LIST_OUTPUT).rstrip('\n').rstrip()

with open(FILE_OUTPUT, 'w', encoding='utf-8', newline='\n') as f:
    f.writelines(TEXT_OUTPUT)
# CODE - FINISH

# TIME END - BEGIN
TIME_END = time.time()
DURATION_IN_SECOND = TIME_END - TIME_START
DAYS = int(DURATION_IN_SECOND // 86400)
DURATION_IN_SECOND %= 86400
HOURS = int(DURATION_IN_SECOND // 3600)
DURATION_IN_SECOND %= 3600
MINUTES = int(DURATION_IN_SECOND // 60)
DURATION_IN_SECOND %= 60
PARTS = []
if DAYS:
    PARTS.append(f'{DAYS} day(s)')
if HOURS:
    PARTS.append(f'{HOURS} hour(s)')
if MINUTES:
    PARTS.append(f'{MINUTES} minute(s)')
if DURATION_IN_SECOND or not PARTS:
    PARTS.append(f'{DURATION_IN_SECOND:.2f} second(s)')
print(' '.join(PARTS))
# TIME END - FINISH

# DONE - BEGIN
print('====\nDONE\n====')
# DONE - FINISH

# ALARM PLAY - BEGIN
import warnings
warnings.filterwarnings('ignore')
import os
os.environ['PYGAME_HIDE_SUPPORT_PROMPT'] = '1'
import pygame
import time
pygame.mixer.init()
pygame.mixer.music.load('alarm.mp3')
for _ in range(3):
    pygame.mixer.music.play()
    while pygame.mixer.music.get_busy():
        time.sleep(0.01)
pygame.mixer.quit()
# ALARM PLAY - FINISH

# 3 - CREATE - REPAIR_TEST_CASE_OUTPUT_EXTENSION.txt - END

0.00 second(s)
====
DONE
====
